<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_OB_2%EC%A1%B0_Project2_%EC%84%B1%EB%8A%A5%ED%96%A5%EC%83%81%2C%ED%9B%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 성능 향상 + 후처리



In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.optimize import minimize
from scipy.sparse import hstack
from sklearn.metrics import log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold

DATA_DIR = '/content/drive/MyDrive/ESAA_OB/OB_data/Project(2)'

train = pd.read_csv(f'{DATA_DIR}/train_clean.csv')
test = pd.read_csv(f'{DATA_DIR}/test_clean.csv')
sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print("데이터 로드 완료:")
print(f"- train: {train.shape}")
print(f"- test: {test.shape}")
print(f"- sample_submission: {sub.shape}")

데이터 로드 완료:
- train: (54739, 5)
- test: (19617, 4)
- sample_submission: (19617, 6)


## 1. TF-IDF 피처 생성

In [ ]:
X_text = train['text_model'].fillna('')
T_text = test['text_model'].fillna('')
y = train['author'].values

TOKEN = r"(?u)\b\w+(?:['’]\w+)?\b|[^\w\s]"
word_kw = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=TOKEN)
char_kw = dict(analyzer='char', ngram_range=(2, 5), min_df=3, sublinear_tf=True)

wv = TfidfVectorizer(dtype=np.float32, **word_kw)
cv = TfidfVectorizer(dtype=np.float32, **char_kw)

# LR용: 단어 + 문자 피처
X_all = hstack([wv.fit_transform(X_text), cv.fit_transform(X_text)]).tocsr()
T_all = hstack([wv.transform(T_text), cv.transform(T_text)]).tocsr()

# NB용: 단어 피처 전용
wv_nb = TfidfVectorizer(dtype=np.float32, **word_kw)
X_word = wv_nb.fit_transform(X_text)
T_word = wv_nb.transform(T_text)

## 2. 5-Fold Cross Validation 및 앙상블 (LR + NB)


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lr = np.zeros((len(y), 5))
oof_nb = np.zeros((len(y), 5))

test_pred_lr = np.zeros((len(T_text), 5))
test_pred_nb = np.zeros((len(T_text), 5))

# 5-Fold 모델 학습 및 OOF 예측
for fold, (tr, va) in enumerate(skf.split(X_all, y), 1):
    # Model 1: Logistic Regression
    m_lr = LogisticRegression(C=10, max_iter=2000).fit(X_all[tr], y[tr])
    oof_lr[va] = m_lr.predict_proba(X_all[va])
    test_pred_lr += m_lr.predict_proba(T_all) / 5

    # Model 2: Naive Bayes
    m_nb = MultinomialNB(alpha=0.05).fit(X_word[tr], y[tr])
    oof_nb[va] = m_nb.predict_proba(X_word[va])
    test_pred_nb += m_nb.predict_proba(T_word) / 5

# Soft Voting 앙상블 (가중치: LR 0.7, NB 0.3)
w_lr, w_nb = 0.7, 0.3
oof_ensemble = (oof_lr * w_lr) + (oof_nb * w_nb)
test_pred_ensemble = (test_pred_lr * w_lr) + (test_pred_nb * w_nb)

print(f"▶ LR OOF LogLoss: {log_loss(y, oof_lr):.4f}")
print(f"▶ NB OOF LogLoss: {log_loss(y, oof_nb):.4f}")
print(f"▶ 앙상블 OOF LogLoss: {log_loss(y, oof_ensemble):.4f}")

▶ LR OOF LogLoss: 0.4459
▶ NB OOF LogLoss: 0.4849
▶ 앙상블 OOF LogLoss: 0.4230


## 3. 앙상블 결과에 131자 이상 기준 Temperature Scaling 적용

In [ ]:
def apply_temperature(probs, t):
    p_safe = np.clip(probs, 1e-15, 1 - 1e-15)
    logits = np.log(p_safe)
    scaled_logits = logits / t
    exp_logits = np.exp(scaled_logits - np.max(scaled_logits, axis=1, keepdims=True))
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

# Test 세트와 유사한 131자 이상 긴 문장만 추출
is_long = (train['text'].str.len() >= 131).values
y_long = y[is_long]
oof_ens_long = oof_ensemble[is_long]

# 앙상블 OOF 중 긴 문장에 최적화된 T 찾기
def loss_func(t):
    calibrated_p = apply_temperature(oof_ens_long, t[0])
    return log_loss(y_long, calibrated_p)

res = minimize(loss_func, [1.0], bounds=[(0.5, 3.0)], method='L-BFGS-B')
best_t = res.x[0]

print(f"▶ 최적 Temperature (T): {best_t:.4f}")
print(f"▶ 후처리 전 긴 문장 LogLoss: {log_loss(y_long, oof_ens_long):.4f}")
print(f"▶ 후처리 후 긴 문장 LogLoss: {loss_func([best_t]):.4f}")

▶ 최적 Temperature (T): 0.6289
▶ 후처리 전 긴 문장 LogLoss: 0.2685
▶ 후처리 후 긴 문장 LogLoss: 0.2300


## 4. 최종 예측 확률 보정 + Clipping

In [ ]:
# 1) 최적 T 적용
final_probs = apply_temperature(test_pred_ensemble, best_t)

# 2) Clipping 적용 (극단적 페널티 방지)
final_probs = np.clip(final_probs, 1e-15, 1 - 1e-15)
final_probs = final_probs / np.sum(final_probs, axis=1, keepdims=True)

# 최종 submission 파일 1개로 저장
sub_final = sub.copy()
sub_final[['0', '1', '2', '3', '4']] = final_probs

SAVE_PATH = f'{DATA_DIR}/submission_ensemble_final.csv'
sub_final.to_csv(SAVE_PATH, index=False)

print(f"최종 submission 파일 생성 완료: {SAVE_PATH}")
display(sub_final.head())

최종 submission 파일 생성 완료: /content/drive/MyDrive/ESAA_OB/OB_data/Project(2)/submission_ensemble_final.csv


,index,0,1,2,3,4
0,0,0.033731,6.498888e-01,0.304538,0.011621,0.000222
1,1,0.012026,9.794718e-01,0.000055,0.007635,0.000812
2,2,0.999294,3.433899e-04,0.000032,0.000197,0.000133
3,3,0.000507,2.378776e-07,0.996669,0.000008,0.002816
4,4,0.989839,6.453424e-03,0.001114,0.002262,0.000331
